# Week 6: Advanced Retrieval, Transform and Measure

## Purpose
Most retrieval failures are query failures: the user's words do not match the corpus vocabulary, so the right document never surfaces. This week I work on the query side and on the ranking side, and prove whether it helped. 

Learning goals
* Apply a query transformation (multi-query) on top of the existing retriever.
* Add a re-ranking step (RRF) and measure its effect.
* Measure retrieval quality before and after, per query, and find where a transformation backfires.

## What's in this notebook
* **Part 0 — Rebuild the Week 5 pipeline:** chunking strategies, embedders (local MiniLM + hosted `text-embedding-3-small`), the pgvector connection, the baseline single-query retriever, and the Week 5 query set with full and primary relevance labels.
* **Part 1 — Transform and re-rank:** adds BM25 lexical search, an LLM-based multi-query generator, Reciprocal Rank Fusion (RRF), and a Jev System One reranker; wires all four into an extended `retrieve()` and updates the scoring functions to match.
* **Part 2 — Measure query transformation + RRF:** baseline single-query vector search vs. multi-query + RRF at `k=3`, with per-query deltas, aggregate precision/recall/hit-rate, and latency.
* **Part 3 — Find a failure and explain it:** looks for a query where multi-query + RRF hurt retrieval quality, and explains why none regressed in this sample.
* **Part 4 — Jev reranker exploration:** two experiments — (1) cosine similarity vs. Jev-reranked top-1 on pure vector search, and (2) vector+Jev vs. full hybrid + multi-query + RRF + Jev — each with deltas, aggregates, and latency.

In [1]:
# Imports
from dataclasses import dataclass, field
import re, yaml
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pathlib import Path
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import hashlib, json, psycopg, pgvector
from pgvector.psycopg import register_vector
from psycopg import sql
import pandas as pd


c:\Users\brett\projects\github\bretttay24\cosc-650-applied-llm-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. RAG Pipeline from Week 5
### Build the pipeline

### 0a. Chunk Class / Embedder Class

In [2]:
@dataclass
class Chunk:
    text: str
    source: str          # week-5/Docs/Neo4j.md
    chunker: str         # "md-header", "fixed-320-40", ...
    ordinal: int         # position within the doc for this chunker
    header_path: str     # "Microsoft Agent Framework Overview > Install > Python"
    meta: dict = field(default_factory=dict)   # frontmatter + anything else


@dataclass
class Embedder:
    name: str
    dim: int
    encode: callable   

#### Chunking Strategies

In [3]:

# ===================== Chunking Strategies ===================== 

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

def header_path_of(section) -> str:
    return " > ".join(section.metadata[level] for level in ("h1", "h2", "h3") if section.metadata.get(level))

# === Strategy 1 ===

def chunk_fixed(meta, body, source, size=320, overlap=40):
    """Baseline: header-blind character windows, whitespace collapsed."""
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    text = re.sub(r"\s+", " ", body).strip()
    chunker_name = f"fixed-{size}-{overlap}"
    return [
        Chunk(text=text[start:start + size],
              source=source,
              chunker=chunker_name,
              ordinal=ordinal,
              header_path="",
              meta={**meta, "char_start": start})
        for ordinal, start in enumerate(range(0, len(text), size - overlap))
    ]


# === Strategy 2 ===

def chunk_markdown_header(meta, body, source):
    """One chunk per markdown section; sections are unbounded in length."""
    sections = MarkdownHeaderTextSplitter(HEADERS, strip_headers=False).split_text(body)
    return [
        Chunk(text=section.page_content,
              source=source,
              chunker="md-header",
              ordinal=ordinal,
              header_path=header_path_of(section),
              meta={**meta, **section.metadata})
        for ordinal, section in enumerate(sections)
    ]


# === Strategy 3 ===

def chunk_markdown_header_recursive(meta, body, source, size=800, overlap=100):
    """Header split first, then bound section length; sections here run long."""
    sections = MarkdownHeaderTextSplitter(HEADERS, strip_headers=False).split_text(body)
    sub_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)

    chunks = []
    for section in sections:
        for piece in sub_splitter.split_text(section.page_content):
            chunks.append(
                Chunk(text=piece,
                      source=source,
                      chunker="md-header+rec",
                      ordinal=len(chunks),
                      header_path=header_path_of(section),
                      meta={**meta, **section.metadata})
            )
    return chunks

CHUNKERS = {
    "fixed-320-40": chunk_fixed,
    "md-header": chunk_markdown_header,
    "md-header+rec": chunk_markdown_header_recursive,
}



#### Embeddings

In [4]:

# ===================== Embeddings ===================== 

def _normalize(vectors: np.ndarray) -> np.ndarray:
    vectors = np.asarray(vectors, dtype="float32")
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

# === 1 - Local: sentence-transformers ===
#=========================================
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm = Embedder(
    name="all-MiniLM-L6-v2",
    dim=minilm_model.get_embedding_dimension(),
    encode=lambda texts: minilm_model.encode(texts, normalize_embeddings=True, batch_size=64).astype("float32"),
)



# === 2 - Hosted: Azure AI Foundry, text-embedding-3-small ===
#=============================================================
load_dotenv()

project = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

# Embeddings route on the resource root; the project-scoped /openai/v1/ path has no /embeddings.
RESOURCE_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]


openai_client = AzureOpenAI(
    azure_endpoint=RESOURCE_ENDPOINT,
    azure_ad_token_provider=get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    ),
    api_version="2024-10-21",
)

# On Foundry this is the deployment name, which need not match the model name.
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")


def _openai_encode(texts, model=EMBEDDING_DEPLOYMENT, dimensions=None, batch_size=256):
    """
    Encode a list of texts into embeddings using the OpenAI API via Azure AI Foundry.

    Args:
        texts (List[str]): The texts to encode.
        model (str): The embedding model to use.
        dimensions (int, optional): The dimensionality of the embeddings.
        batch_size (int): The batch size for API requests.

    Returns:
        np.ndarray: Normalized embeddings as a NumPy array.
    """
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        kwargs = {"dimensions": dimensions} if dimensions else {}
        response = openai_client.embeddings.create(model=model, input=batch, **kwargs)
        # The API may return items out of order; index is authoritative.
        vectors.extend(item.embedding for item in sorted(response.data, key=lambda item: item.index))
    return _normalize(np.array(vectors))


openai_small = Embedder(
    name="text-embedding-3-small",
    dim=1536,
    encode=_openai_encode,
)

EMBEDDERS = [openai_small, minilm] 
EMBEDDING_TABLES = {
    "all-MiniLM-L6-v2": "chunk_emb_minilm",
    "text-embedding-3-small": "chunk_emb_oai",
}

def embedding_table(embedder):
    """Return the pgvector table associated with an embedder."""
    return sql.Identifier(EMBEDDING_TABLES[embedder.name])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9082.59it/s]


### 0b. Connect to the local pgvector database populated in Week 5

In [5]:
# Kept open for the rest of the notebook; close it in the last cell.
conn = psycopg.connect(os.environ["PGVECTOR_URL"], row_factory=psycopg.rows.dict_row)
register_vector(conn)

### 0c. Baseline retriever

In [6]:
def retrieve(connection, query, embedders=None, chunkers=None, k=5):
    """
    Retrieve the top-k chunks for a query from one or more embedders' vector tables.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        query (str): The natural-language query.
        embedders (str | Embedder | list | None): Registered name(s) such as
            "all-MiniLM-L6-v2" or "text-embedding-3-small", Embedder objects, or None for all.
        chunkers (str | list[str] | None): Restrict to one or more chunking strategies, or None for all.
        k (int): The number of chunks to return per embedder.

    Returns:
        dict[str, list[dict]]: Embedder name -> rows of chunk_uid, source, chunker,
            header_path, content, and distance, ordered by ascending cosine distance.

    Examples:
        ```
        q = "What package do I install to use Agent Hooks?"

        retrieve(conn, q)                                                  # both models, all chunkers
        retrieve(conn, q, embedders="text-embedding-3-small")              # hosted only
        retrieve(conn, q, embedders="all-MiniLM-L6-v2", chunkers="md-header")
        retrieve(conn, q, embedders=["all-MiniLM-L6-v2", "text-embedding-3-small"],
                chunkers=["md-header", "md-header+rec"])

        rows = retrieve(conn, q, embedders="all-MiniLM-L6-v2")["all-MiniLM-L6-v2"]
        ```
    """
    by_name = {embedder.name: embedder for embedder in EMBEDDERS}

    if embedders is None:
        selected = list(EMBEDDERS)
    else:
        if isinstance(embedders, (str, Embedder)):
            embedders = [embedders]
        selected = []
        for item in embedders:
            if isinstance(item, Embedder):
                selected.append(item)
            elif item in by_name:
                selected.append(by_name[item])
            else:
                raise KeyError(f"unknown embedder {item!r}; registered: {sorted(by_name)}")

    if isinstance(chunkers, str):
        chunkers = [chunkers]
    elif chunkers is not None:
        chunkers = list(chunkers)

    results = {}
    for embedder in selected:
        statement = sql.SQL("""
            SELECT c.chunk_uid, c.source, c.chunker, c.header_path, c.content,
                   e.emb <=> %(vector)s AS distance
            FROM {table} e
            JOIN doc_chunk c USING (chunk_uid)
            WHERE %(chunkers)s::text[] IS NULL OR c.chunker = ANY(%(chunkers)s::text[])
            ORDER BY distance
            LIMIT %(k)s
        """).format(table=embedding_table(embedder))

        params = {"vector": embedder.encode([query])[0], "chunkers": chunkers, "k": k}
        with connection.cursor() as cursor:
            results[embedder.name] = cursor.execute(statement, params).fetchall()
    return results

### 0d. Week 5 query set and relevance labels

#### Query Set

In [7]:
QA = [
    {"query": "What function lets an agent search a vector store?",
     "fact": "create_vector_search_tool",
     "source": "week-5/Docs/Agent-Framework-RAG.md"},

    {"query": "How do I turn on hybrid keyword search for a search tool?",
     "fact": "keyword_hybrid",
     "source": "week-5/Docs/Agent-Framework-RAG.md"},

    {"query": "What package do I install to use Agent Hooks?",
     "fact": "agent-hooks-sdk",
     "source": "week-5/Docs/Agent-hooks.md"},

    {"query": "How long does an Agent Hooks interceptor wait before it times out?",
     "fact": "five seconds",
     "source": "week-5/Docs/Agent-hooks.md"},

    {"query": "Which modes of retrieval are supported by AzureAISearchContextProvider?",
     "fact": "semantic and agentic retrieval",
     "source": "week-5/Docs/Azure-AI-Search.md"},

    {"query": "Which Cosmos DB provider contains memories of facts or summaries between sessions?",
     "fact": "CosmosMemoryContextProvider",
     "source": "week-5/Docs/Azure-Cosmos-DB.md"},

    {"query": "Which evaluator runs checks locally without calling an API?",
     "fact": "LocalEvaluator",
     "source": "week-5/Docs/Evaluation.md"},

    {"query": "Which agent comes with planning and todo tracking built in?",
     "fact": "Harness Agent",
     "source": "week-5/Docs/Microsoft-Agent-Framework-Overview.md"},

    {"query": "What should I pin so evaluation results stay comparable across runs?",
     "fact": "datasets, model deployments, evaluator versions, and rubric versions",
     "source": "week-5/Docs/Microsoft-Foundry-evaluation.md"},

    {"query": "What package do I install for the Neo4j GraphRAG context provider?",
    "fact": "agent-framework-neo4j",
    "source": "week-5/Docs/Neo4j.md"},
]

qa = [(item["query"], item["fact"]) for item in QA]

#### Relevance Labels (Partially Relevant and Primarily Relevant Present)

In [8]:
# Relevant = every md-header chunk in the source doc that contains the fact string.
RELEVANT_SQL = """
SELECT chunk_uid, header_path
FROM doc_chunk
WHERE chunker = 'md-header'
  AND source  = %(source)s
  AND strpos(lower(content), lower(%(fact)s)) > 0
"""

def relevant_rows(connection, item):
    """Same query as relevant_chunks, kept as rows so header_path stays visible."""
    with connection.cursor() as cursor:
        return cursor.execute(RELEVANT_SQL, {"source": item["source"], "fact": item["fact"]}).fetchall()

RELEVANT = {
    item["query"]: [(row["chunk_uid"], row["header_path"]) for row in relevant_rows(conn, item)]
    for item in QA
}

relevant_df = pd.DataFrame(
    [(query[:55], chunk_uid[:5] + "...", header_path)
     for query, rows in RELEVANT.items()
     for chunk_uid, header_path in rows],
    columns=["query", "chunk_uid", "header_path"],
)
relevant_df["query"] = relevant_df["query"].mask(relevant_df["query"].duplicated(), "")  # blank repeats, like a grouped table
print("==== Relevant score card ====")
print()
relevant_df

==== Relevant score card ====



,query,chunk_uid,header_path
0,What function lets an agent search a vector st...,e0c35...,RAG | Microsoft Learn
1,,b9d1b...,RAG | Microsoft Learn > Create a native vector...
2,,168f5...,RAG | Microsoft Learn > Customize search behavior
3,How do I turn on hybrid keyword search for a s...,168f5...,RAG | Microsoft Learn > Customize search behavior
4,What package do I install to use Agent Hooks?,0f2ae...,Agent hooks | Microsoft Learn > Install Agent ...
5,How long does an Agent Hooks interceptor wait ...,33018...,Agent hooks | Microsoft Learn > Configure enfo...
6,Which modes of retrieval are supported by Azur...,9615a...,Azure AI Search | Microsoft Learn
7,Which Cosmos DB provider contains memories of ...,0813c...,Azure Cosmos DB | Microsoft Learn > Add long-t...
8,,564c7...,Azure Cosmos DB | Microsoft Learn
9,Which evaluator runs checks locally without ca...,8563d...,Evaluation | Microsoft Learn > Core concepts


#### Primary Relevant 

In [9]:
# Below I make a list of the primary relevant header paths in my documents for each query.
PRIMARY_RELEVANT = {
    "What function lets an agent search a vector store?":
        "RAG | Microsoft Learn",
    "How do I turn on hybrid keyword search for a search tool?":
        "RAG | Microsoft Learn > Customize search behavior",
    "What package do I install to use Agent Hooks?":
        "Agent hooks | Microsoft Learn > Install Agent Hooks",
    "How long does an Agent Hooks interceptor wait before it times out?":
        "Agent hooks | Microsoft Learn > Configure enforcement",
    "Which modes of retrieval are supported by AzureAISearchContextProvider?":
        "Azure AI Search | Microsoft Learn",
    "Which Cosmos DB provider contains memories of facts or summaries between sessions?":
        "Azure Cosmos DB | Microsoft Learn",
    "Which evaluator runs checks locally without calling an API?":
        "Evaluation | Microsoft Learn > Local evaluators",
    "Which agent comes with planning and todo tracking built in?":
        "Microsoft Agent Framework Overview | Microsoft Learn",
    "What should I pin so evaluation results stay comparable across runs?":
        "Microsoft Foundry evaluation | Microsoft Learn > Quality gates",
    "What package do I install for the Neo4j GraphRAG context provider?":
        "Neo4j | Microsoft Learn > GraphRAG from an existing knowledge graph > Installation",
}

def primary_chunk_uid(query):
    """Resolve chunk_uid for a query's single 'Relevant' header_path, via the existing RELEVANT rows."""
    target = PRIMARY_RELEVANT[query]
    return next(uid for uid, header_path in RELEVANT[query] if header_path == target)


## Part 1: Transform and re-rank
Extend the Week 5 retriever with one query transformation and a re-ranking step over the retrieved candidates.

### Add  BM25 lexical search to pipeline

* I decided I want to try adding BM25 (lexical search) to my dense vector (semantic search) to create a hybrid search method. I will add this to my retrieval pipeline and I will add a RRF that fuses the resutls of the two search methods. 

In [10]:
from rank_bm25 import BM25Okapi


_BM25_CACHE = {}
_BM25_TOKEN = re.compile(r"[A-Za-z0-9]+(?:[-_./][A-Za-z0-9]+)*")
_CAMEL_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")


def _tokenize_bm25(text):
    """Tokenize prose and preserve useful technical-identifier forms."""
    tokens = []

    for raw_token in _BM25_TOKEN.findall(text or ""):
        whole_token = raw_token.lower()
        tokens.append(whole_token)

        # CosmosMemoryContextProvider -> cosmos, memory, context, provider
        # agent-framework-neo4j -> agent, framework, neo4j
        expanded = _CAMEL_BOUNDARY.sub(" ", raw_token)
        parts = re.split(r"[-_./\s]+", expanded)

        for part in parts:
            normalized_part = part.lower()
            if normalized_part and normalized_part != whole_token:
                tokens.append(normalized_part)

    return tokens


def _get_bm25_index(connection, chunkers=None):
    """Build and cache one BM25 index for each chunker selection."""
    if isinstance(chunkers, str):
        chunkers = [chunkers]

    cache_key = (
        None
        if chunkers is None
        else tuple(sorted(set(chunkers)))
    )

    if cache_key not in _BM25_CACHE:
        with connection.cursor() as cursor:
            rows = cursor.execute(
                """
                SELECT chunk_uid, source, chunker, header_path, content
                FROM doc_chunk
                WHERE %(chunkers)s::text[] IS NULL
                   OR chunker = ANY(%(chunkers)s::text[])
                ORDER BY chunk_uid
                """,
                {
                    "chunkers": (
                        list(cache_key)
                        if cache_key is not None
                        else None
                    )
                },
            ).fetchall()

        documents = [dict(row) for row in rows]

        tokenized_corpus = [
            # Duplicate header tokens to give headings additional weight.
            _tokenize_bm25(document["header_path"]) * 2
            + _tokenize_bm25(document["content"])
            for document in documents
        ]

        model = BM25Okapi(tokenized_corpus) if documents else None
        _BM25_CACHE[cache_key] = (model, documents)

    return _BM25_CACHE[cache_key]


def clear_bm25_cache():
    """Force indexes to rebuild after doc_chunk changes."""
    _BM25_CACHE.clear()


def search_bm25(connection, query, k=10, chunkers=None):
    """Return top-k PostgreSQL chunks ranked with Okapi BM25."""
    if k < 1:
        raise ValueError("k must be at least 1")

    model, documents = _get_bm25_index(connection, chunkers)
    query_tokens = _tokenize_bm25(query)

    if model is None or not query_tokens:
        return []

    scores = model.get_scores(query_tokens)

    ranked_indices = sorted(
        (
            index
            for index, score in enumerate(scores)
            if score > 0
        ),
        key=lambda index: (
            -float(scores[index]),
            documents[index]["chunk_uid"],
        ),
    )[:k]

    return [
        {
            **documents[index],
            "retriever": "bm25",
            "search_query": query,
            "retrieval_rank": rank,
            "raw_score": float(scores[index]),
        }
        for rank, index in enumerate(ranked_indices, start=1)
    ]

### Create a Function to Generate Multiple Queries
- This is based off of the Code from rag-fusion github. This uses diverse queries. 
    * https://github.com/Raudaschl/rag-fusion/blob/master/main.py
- I have added the ability to have 'N' - number of queries and I am also putting this to my model MS Foundry

In [11]:
from prompts.loader import load
import re

_PREAMBLE = re.compile(r"^(here (are|is)|sure[,!]|okay|below are)", re.IGNORECASE)
_BULLET = re.compile(r"^\s*(?:[-*•–]|\d+[.)]|\(\d+\))\s*")

def generate_multiple_queries(original_query, num_queries):
    """ 
    Generate multiple variations of the original query.

    Args:
        original_query (str): The original query string.
        num_queries (int): The number of query variations to generate.

    Returns:
        list: A list containing the original query and its variations.
    """
    prompt = load("multiple-query-generator")
    rendered_prompt = prompt.text.format(
        N=num_queries,
        original_query=original_query,
    )

    response = project.get_openai_client().responses.create(
        model=prompt.model,
        input=rendered_prompt,
        **prompt.params,
    )

    return _parse_queries(response.output_text)[:num_queries]


def _parse_queries(raw):
    """Clean newline-separated LLM queries and remove common list formatting."""
    if not raw:
        return []

    lines = raw.splitlines() if isinstance(raw, str) else raw
    queries = []

    for line in lines:
        text = line.strip()
        if not text or _PREAMBLE.match(text) or text.endswith(":"):
            continue

        text = _BULLET.sub("", text).strip().strip("*").strip().strip('"').strip()
        if text:
            queries.append(text)

    return queries



### Create a Reciprocal Rerank Fusion (RRF) Function 
This is a rewrite based on the reciprocal rank fusion function found in the Rag fusion repository.
- https://github.com/Raudaschl/rag-fusion/blob/master/main.py


In [12]:
def reciprocal_rank_fusion(retrieved_chunks, rank_constant=60, top_k=None):

    """
    Fuse rankings from multiple queries and retrievers.
    
    Args:
        retrieved_chunks (list): A list of dictionaries containing retrieved chunks.
        rank_constant (int, optional): The constant used in the reciprocal rank fusion formula. Defaults to 60.
        top_k (int, optional): The number of top-ranked chunks to return. If None, return all. Defaults to None.
    Returns:
        list: A list of fused and ranked chunks, each containing the original chunk data along with the fused rank and RRF score.
    """
    if rank_constant <= 0:
        raise ValueError("rank_constant must be greater than 0")

    # Each retriever/query combination is one independent ranked list.
    query_rankings = {}

    for candidate in retrieved_chunks:
        ranking_key = (
            candidate["retriever"],
            candidate["search_query"],
        )
        query_rankings.setdefault(ranking_key, []).append(candidate)

    fused_chunks = {}

    for (retriever, search_query), ranking_candidates in query_rankings.items():
        ranked_candidates = sorted(
            ranking_candidates,
            key=lambda candidate: candidate["retrieval_rank"],
        )

        for candidate in ranked_candidates:
            chunk_uid = candidate["chunk_uid"]

            if chunk_uid not in fused_chunks:
                # Remove fields that describe only one retrieval occurrence.
                fused_chunks[chunk_uid] = {
                    key: value
                    for key, value in candidate.items()
                    if key not in {
                        "retriever",
                        "search_query",
                        "retrieval_rank",
                        "raw_score",
                        "distance",
                    }
                }
                fused_chunks[chunk_uid]["rrf_score"] = 0.0
                fused_chunks[chunk_uid]["retrievals"] = []

            rank = candidate["retrieval_rank"]
            fused_chunks[chunk_uid]["rrf_score"] += (
                1 / (rank_constant + rank)
            )

            fused_chunks[chunk_uid]["retrievals"].append({
                "retriever": retriever,
                "search_query": search_query,
                "rank": rank,
                "raw_score": candidate["raw_score"],
            })

    ranked_chunks = sorted(
        fused_chunks.values(),
        key=lambda candidate: (
            -candidate["rrf_score"],
            candidate["chunk_uid"],
        ),
    )

    if top_k is not None:
        ranked_chunks = ranked_chunks[:top_k]

    for fused_rank, candidate in enumerate(ranked_chunks, start=1):
        candidate["fused_rank"] = fused_rank

    return ranked_chunks

### Create a Jev ReRanker
Jev takes a query and a candidate chunk and returns a single `noul` value — the probability that the answer to a yes/no question is "yes" (0 = no, 1 = yes). Here the question is whether the chunk is relevant to the query, so a `noul` close to 1 means Jev is confident the chunk answers the query, and a value close to 0 means it's confident it doesn't. Given a pool of candidates (either the top-20 from similarity search alone, or the top-20 after RRF fusion), Jev scores every chunk in the pool against the query this way, and the pool is reranked by that probability instead of by cosine similarity or RRF score.

- [Rag Fusion Jev Eperiment](https://github.com/Raudaschl/rag-fusion/tree/master/experiments/jev-in-the-pipeline)
- [Rag Fusion Jev Eperiment- Youtube Video](https://www.youtube.com/watch?v=SzfGTXWZf4o)
- [Jev Reranker by hev](https://github.com/hev/reranker)
- [System One overview](https://docs.typesafe.ai/concepts/system-one) — what a System One model is and how it works
- [Noul primitive reference](https://docs.typesafe.ai/primitives/noul) — the yes/no question type used here


In [13]:
from typesafe_sdk import Noul, NoulCriteria, RetryPolicy, TypeSafeClient

JEV_QUESTION = (
    "Document `documents.{id}` is relevant to `query`: "
    "it contains information that answers or directly addresses it."
)

JEV_CRITERIA = {
    "true": (
        "The document contains information that answers the query "
        "or directly addresses what it asks about."
    ),
    "false": (
        "The document is only loosely related, on a similar topic, "
        "or does not address what the query asks."
    ),
}

jev_client = TypeSafeClient(
    retry=RetryPolicy(
        max_retries=6,
        backoff_max=30.0,
        timeout=90.0,
    )
)


def rerank_with_jev(
    query,
    candidates,
    *,
    top_n=None,
    threshold=None,
    client=jev_client,
    model="jev-latest",
):
    """Rerank retrieved candidate dictionaries by P(relevant).
    Args:
        query (str): The query string to match against candidate documents.
        candidates (list[dict]): A list of candidate document dictionaries.
        top_n (int, optional): The maximum number of top candidates to return.
        threshold (float, optional): Minimum JEV score to include a candidate.
        client (TypeSafeClient, optional): The JEV client to use.
        model (str, optional): The JEV model to use.
    
    Returns:
        list[dict]: The reranked list of candidate dictionaries with added
        "jev_score" and "jev_rank" fields.
    """
    if not candidates:
        return []

    if len(candidates) > 30:
        raise ValueError(
            "This notebook adapter supports at most 30 candidates per call"
        )

    candidate_ids = {
        f"D{index:02d}": candidate
        for index, candidate in enumerate(candidates)
    } # stable short ids so Jev can reference one candidate without echoing full chunk text

    state = {
        "query": query,
        "documents": {
            candidate_id: {
                "title": candidate.get("header_path", ""),
                "text": candidate["content"],
            }
            for candidate_id, candidate in candidate_ids.items()
        },
    } # every Noul question below reads from this same state via its candidate_id

    questions = {
        candidate_id: Noul(
            instructions=JEV_QUESTION.format(id=candidate_id),
            criteria=NoulCriteria(**JEV_CRITERIA),
        )
        for candidate_id in candidate_ids
    } # one yes/no question per candidate, all sent in a single request and scored in parallel

    response = client.system_one(
        state=state,
        questions=questions,
        model=model,
    )

    reranked = []
    for candidate_id, candidate in candidate_ids.items():
        jev_score = float(response.answers[candidate_id].noul) # P(relevant) in [0, 1] for this candidate

        if threshold is not None and jev_score < threshold:
            continue

        reranked.append({
            **candidate,
            "jev_score": jev_score,
        })

    reranked.sort(
        key=lambda candidate: (
            -candidate["jev_score"],
            -candidate.get("rrf_score", 0.0),
            candidate["chunk_uid"],
        )
    )

    for rank, candidate in enumerate(reranked, start=1):
        candidate["jev_rank"] = rank

    return reranked[:top_n] if top_n is not None else reranked

### Add to the Query Transformation and RRF to Last Weeks retrieve

In [14]:
def retrieve(
    connection,
    query,
    embedders=None,
    chunkers=None,
    retrieval_k=10,
    pool_k=20,
    final_k=5,
    hybrid_search=False,
    multiple_query=False,
    additional_query_count=0,
    jev_rerank=False,
):
    """
    Retrieve the top-k chunks for a query from one or more embedders' vector tables.

    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        query (str): The natural-language query.
        embedders (str | Embedder | list | None): Registered name(s) such as
            "all-MiniLM-L6-v2" or "text-embedding-3-small", Embedder objects, or None for all.
        chunkers (str | list[str] | None): Restrict to one or more chunking strategies, or None for all.
        retrieval_k (int, optional): The number of chunks to retrieve per query. Defaults to 10.
        pool_k (int, optional): The number of top candidates to consider for fusion. Defaults to 20.
        final_k (int, optional): The number of final results to return after fusion. Defaults to 5.
        hybrid_search (bool, optional): Whether to combine BM25 and vector search results. Defaults to False.
        multiple_query (bool, optional): Whether to generate multiple diverse queries. Defaults to False.
        additional_query_count (int, optional): The number of **additional** queries to generate. Defaults to 0.
        jev_rerank (bool, optional): Whether to apply JEV reranking. Defaults to False.

    Returns:
        dict[str, list[dict]]: Embedder name -> rows of chunk_uid, source, chunker,
            header_path, content, and distance, ordered by ascending cosine distance.

    Examples:
        ```
        q = "What package do I install to use Agent Hooks?"
        ```
    """
    by_name = {embedder.name: embedder for embedder in EMBEDDERS}

    if embedders is None:
        selected = list(EMBEDDERS)
    else:
        if isinstance(embedders, (str, Embedder)):
            embedders = [embedders]
        selected = []
        for item in embedders:
            if isinstance(item, Embedder):
                selected.append(item)
            elif item in by_name:
                selected.append(by_name[item])
            else:
                raise KeyError(f"unknown embedder {item!r}; registered: {sorted(by_name)}")

    if isinstance(chunkers, str):
        chunkers = [chunkers]
    elif chunkers is not None:
        chunkers = list(chunkers)

    # =================== ADD the Multiple Query Transformation ==================
    search_queries = [query]

    if multiple_query:
        if additional_query_count < 1:
            raise ValueError("additional_query_count must be at least 1")

        search_queries.extend(
            generate_multiple_queries(query, additional_query_count)
        )

            # === Display multiple queries ===
    # print(f"Original query:\n   - {query}")
    # print("Additional diverse queries:")
    # for number, search_query in enumerate(search_queries[1:], start=1):
    #     print(f"  {number}. {search_query}")


    # ================= ADD BM25 if hybrid_search is enabled =================
    # BM25 does not depend on the embedding model, so run it once.
    bm25_by_query = {}

    if hybrid_search:
        bm25_by_query = {
            search_query: search_bm25(
                connection,
                search_query,
                k=retrieval_k,
                chunkers=chunkers,
            )
            for search_query in search_queries
        }

    # ================= Collect results for each embedder =================
    results = {}

    for embedder in selected:
        # Build the vector SQL statement once for this embedding table.
        statement = sql.SQL("""
            SELECT c.chunk_uid, c.source, c.chunker, c.header_path, c.content,
                   e.emb <=> %(vector)s AS distance
            FROM {table} e
            JOIN doc_chunk c USING (chunk_uid)
            WHERE %(chunkers)s::text[] IS NULL OR c.chunker = ANY(%(chunkers)s::text[])
            ORDER BY distance
            LIMIT %(k)s
        """).format(table=embedding_table(embedder))

        # ==== Collect each query's ranked candidates for this embedder ====
        candidates = []

        for search_query in search_queries:
            # print(f"Searching: {search_query!r}")
            params = {
                "vector": embedder.encode([search_query])[0],
                "chunkers": chunkers,
                "k": retrieval_k,  # Not the removed k parameter
            }

            with connection.cursor() as cursor:
                vector_rows = cursor.execute(statement, params).fetchall()

                
            # Convert the raw vector rows into candidate dictionaries with additional metadata.
            for rank, row in enumerate(vector_rows, start=1):
                candidate = dict(row)
                candidate["retriever"] = "vector"
                candidate["search_query"] = search_query
                candidate["retrieval_rank"] = rank
                candidate["raw_score"] = -float(candidate["distance"])
                candidates.append(candidate)

            
            if hybrid_search:
                candidates.extend(bm25_by_query[search_query]) # Add BM25 results to the candidate list

        # ==== ADD:Fuse per-query rankings when multi-query is enabled ====
        if hybrid_search or multiple_query:
            # Fuse the rankings of candidates from multiple queries or hybrid search
            candidates = reciprocal_rank_fusion(
                candidates,
                rank_constant=60,
                top_k=pool_k,
            )
        else:
            # If not using hybrid search or multiple query, just take the top pool_k candidates
            candidates = candidates[:pool_k]

        # ==== Apply JEV reranking if enabled ====
        if jev_rerank:
            candidates = rerank_with_jev(
                query,
                candidates,
                top_n=final_k,
            )
        else:
            # return the top final_k candidates if JEV reranking is not applied
            candidates = candidates[:final_k]    

        result_name = embedder.name

        results[result_name] = candidates

    return results

#### Update Scoring Functions to Adjust for Changes in retrieve()

In [15]:
from time import perf_counter

def score_query_primary_chunk(
    connection,
    item,
    *,
    retrieval_k=10,
    pool_k=20,
    final_k=5,
    chunker="md-header",
    embedders=None,
    hybrid_search=False,
    multiple_query=False,
    additional_query_count=0,
    jev_rerank=False,
):
    """
    Score one query for the configured retrieval pipeline.
    Args:
        connection (psycopg.Connection): An open connection to the pgvector database.
        item (dict): A dictionary containing the query and other metadata.
        retrieval_k (int, optional): The number of chunks to retrieve per query. Defaults to 10.
        pool_k (int, optional): The number of top candidates to consider for fusion. Defaults to 20.
        final_k (int, optional): The number of final results to return after fusion. Defaults to 5.
        chunker (str, optional): The chunking strategy to use. Defaults to "md-header".
        embedders (str | Embedder | list | None, optional): The embedders to use. Defaults to None.
        hybrid_search (bool, optional): Whether to combine BM25 and vector search results. Defaults to False.
        multiple_query (bool, optional): Whether to generate multiple diverse queries. Defaults to False.
        additional_query_count (int, optional): The number of additional queries to generate. Defaults to 0.
        jev_rerank (bool, optional): Whether to apply JEV reranking. Defaults to False.

    Returns:
        list[dict]: A list of dictionaries containing retrieval metrics for the query, including precision, recall, and primary chunk rank.

    """

    relevant_uids = {
        uid for uid, _ in RELEVANT[item["query"]]
    }
    primary_uid = primary_chunk_uid(item["query"])

    start_time = perf_counter()
    retrieved = retrieve(
        connection,
        item["query"],
        embedders=embedders,
        chunkers=chunker,
        retrieval_k=retrieval_k,
        pool_k=pool_k,
        final_k=final_k,
        hybrid_search=hybrid_search,
        multiple_query=multiple_query,
        additional_query_count=additional_query_count,
        jev_rerank=jev_rerank,
    )
    latency_ms = (perf_counter() - start_time) * 1000

    query_mode = (
        "multi-query"
        if multiple_query
        else "single-query"
    )

    retrieval_mode = (
        "hybrid"
        if hybrid_search
        else "vector"
    )

    # In the current retrieve function, either hybrid search or
    # multiple-query retrieval causes RRF to run.
    rrf_used = hybrid_search or multiple_query

    if rrf_used and jev_rerank:
        ranking_mode = "RRF -> Jev"
    elif rrf_used:
        ranking_mode = "RRF"
    elif jev_rerank:
        ranking_mode = "Jev"
    else:
        ranking_mode = "cosine similarity"


    rows = []

    for embedder_name, retrieved_chunks in retrieved.items():
        ranked_uids = [
            candidate["chunk_uid"]
            for candidate in retrieved_chunks
        ]
        retrieved_uids = set(ranked_uids)
        hits = retrieved_uids & relevant_uids

        primary_rank = next(
            (
                rank
                for rank, uid in enumerate(ranked_uids, start=1)
                if uid == primary_uid
            ),
            None,
        )

        rows.append({
            "query": item["query"],
            "embedder": embedder_name,
            "query_mode": query_mode,
            "retrieval_mode": retrieval_mode,
            "ranking_mode": ranking_mode,
            "latency_ms": latency_ms,
            "retrieval_k": retrieval_k,
            "pool_k": pool_k,
            "final_k": final_k,
            "n_relevant": len(relevant_uids),
            "n_retrieved": len(retrieved_uids),
            "n_hits": len(hits),
            "precision": (
                len(hits) / len(retrieved_uids)
                if retrieved_uids else 0.0
            ),
            "recall": (
                len(hits) / len(relevant_uids)
                if relevant_uids else 0.0
            ),
            "hit_primary": primary_rank is not None,
            "primary_rank": primary_rank,
            "reciprocal_rank": (
                1 / primary_rank if primary_rank else 0.0
            ),
        })

    return rows


def score_all_queries_primary_chunk(
    connection,
    qa_items=QA,
    **scoring_options,
):
    """Score all queries using one retrieval configuration."""
    return [
        row
        for item in qa_items
        for row in score_query_primary_chunk(
            connection,
            item,
            **scoring_options,
        )
    ]

def compare_retrieval_scores(scores, pivot_column="query_mode", before_value="single-query", after_value="multi-query"):
    metrics = [
        "n_hits", "precision", "recall",
        "hit_primary", "reciprocal_rank", "latency_ms",
    ]
    paired = scores.pivot(
        index="query",
        columns=pivot_column,
        values=metrics,
    )

    report = pd.DataFrame(index=paired.index)

    for metric in metrics:
        before = paired[(metric, before_value)]
        after = paired[(metric, after_value)]

        report[f"{metric}_before"] = before
        report[f"{metric}_after"] = after
        if metric == "hit_primary":
            report[f"{metric}_delta"] = after.astype(int) - before.astype(int)
        else:
            report[f"{metric}_delta"] = after - before

    report["quality_change"] = np.select(
        [report["recall_delta"] > 0, report["recall_delta"] < 0],
        ["improved", "declined"],
        default="unchanged",
    )
    return report.reset_index()

## Part 2:  Measure the Effects of Query Tranformation(multiple query generation) and Reranking (RRF)

In [16]:
# === All outcomes come from text-embedding-3-small embedder ===
embedder = "text-embedding-3-small"

# === Baseline Scores of Vector Similarity with Final K=3 ===
baseline_scores_k3 = pd.DataFrame(
    score_all_queries_primary_chunk(
        conn,
        retrieval_k=10,
        chunker="md-header",
        final_k=3,
        embedders=embedder,
        hybrid_search=False,
        multiple_query=False,
        jev_rerank=False
    )
)
# === Multiple Query Scores / RRF with Final K=3 ===
multiple_query_scores_k3 = pd.DataFrame(
    score_all_queries_primary_chunk(
        conn,
        retrieval_k=10,
        pool_k=20,
        final_k=3,
        chunker="md-header",
        embedders=embedder,
        hybrid_search=False,
        multiple_query=True,
        additional_query_count=3,
        jev_rerank=False,
    )
)
# === Comparison of Baseline and Multiple Query Scores with Final K=3 ===
comparison_scores_k3 = pd.concat(
    [baseline_scores_k3, multiple_query_scores_k3],
    ignore_index=True,
)


In [17]:
display(
    comparison_scores_k3[["query", "query_mode", "retrieval_mode", "ranking_mode", "n_relevant", "n_hits", "precision", "recall", "hit_primary"]].sort_values(["query", "query_mode", "retrieval_mode", "ranking_mode"])
)

,query,query_mode,retrieval_mode,ranking_mode,n_relevant,n_hits,precision,recall,hit_primary
11,How do I turn on hybrid keyword search for a s...,multi-query,vector,RRF,1,1,0.333333,1.000000,True
1,How do I turn on hybrid keyword search for a s...,single-query,vector,cosine similarity,1,1,0.333333,1.000000,True
13,How long does an Agent Hooks interceptor wait ...,multi-query,vector,RRF,1,1,0.333333,1.000000,True
3,How long does an Agent Hooks interceptor wait ...,single-query,vector,cosine similarity,1,0,0.000000,0.000000,False
10,What function lets an agent search a vector st...,multi-query,vector,RRF,3,2,0.666667,0.666667,True
0,What function lets an agent search a vector st...,single-query,vector,cosine similarity,3,2,0.666667,0.666667,True
19,What package do I install for the Neo4j GraphR...,multi-query,vector,RRF,2,2,0.666667,1.000000,True
9,What package do I install for the Neo4j GraphR...,single-query,vector,cosine similarity,2,2,0.666667,1.000000,True
12,What package do I install to use Agent Hooks?,multi-query,vector,RRF,1,1,0.333333,1.000000,True
2,What package do I install to use Agent Hooks?,single-query,vector,cosine similarity,1,1,0.333333,1.000000,True


#### Per Query Deltas

In [18]:
per_query_deltas = compare_retrieval_scores(comparison_scores_k3)

In [19]:
display(
    per_query_deltas[
        [
            "query", "n_hits_delta",
            "precision_delta",
            "recall_delta",
            "hit_primary_before", "hit_primary_after",
        ]
    ].round(3)
)

,query,n_hits_delta,precision_delta,recall_delta,hit_primary_before,hit_primary_after
0,How do I turn on hybrid keyword search for a s...,0,0.0,0.0,True,True
1,How long does an Agent Hooks interceptor wait ...,1,0.333333,1.0,False,True
2,What function lets an agent search a vector st...,0,0.0,0.0,True,True
3,What package do I install for the Neo4j GraphR...,0,0.0,0.0,True,True
4,What package do I install to use Agent Hooks?,0,0.0,0.0,True,True
5,What should I pin so evaluation results stay c...,0,0.0,0.0,True,True
6,Which Cosmos DB provider contains memories of ...,1,0.333333,0.5,True,True
7,Which agent comes with planning and todo track...,0,0.0,0.0,True,True
8,Which evaluator runs checks locally without ca...,0,0.0,0.0,True,True
9,Which modes of retrieval are supported by Azur...,0,0.0,0.0,True,True


#### Aggregates at K3

In [20]:
quality_summary_k3 = pd.DataFrame([
    {
        "metric": "Mean precision",
        "before": per_query_deltas["precision_before"].mean(),
        "after": per_query_deltas["precision_after"].mean(),
        "delta": per_query_deltas["precision_delta"].mean(),
    },
    {
        "metric": "Mean recall",
        "before": per_query_deltas["recall_before"].mean(),
        "after": per_query_deltas["recall_after"].mean(),
        "delta": per_query_deltas["recall_delta"].mean(),
    },
    {
        "metric": "Primary-answer hit rate (%, delta in pp)",
        "before": per_query_deltas["hit_primary_before"].mean() * 100,
        "after": per_query_deltas["hit_primary_after"].mean() * 100,
        "delta": per_query_deltas["hit_primary_delta"].mean() * 100,
    },
])

display(quality_summary_k3.round(3))

,metric,before,after,delta
0,Mean precision,0.400,0.467,0.067
1,Mean recall,0.757,0.907,0.150
2,"Primary-answer hit rate (%, delta in pp)",90.000,100.000,10.000


#### Latency Comparison

In [21]:
latency_columns = [
    "query",
    "retrieval_mode",
    "latency_ms",
]

latency_summary_k3 = (
    comparison_scores_k3
    .groupby(["query_mode", "ranking_mode"], as_index=False)
    .agg(
        mean_ms=("latency_ms", "mean"),
        median_ms=("latency_ms", "median"),
        min_ms=("latency_ms", "min"),
        max_ms=("latency_ms", "max"),
        p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
    )
)

display(latency_summary_k3.round(2))

,query_mode,ranking_mode,mean_ms,median_ms,min_ms,max_ms,p95_ms
0,multi-query,RRF,3823.57,3451.63,3107.96,5421.69,5390.10
1,single-query,cosine similarity,443.38,160.85,130.20,2890.49,1725.57


#### Analysis: 
##### **Top 3 Vector Search vs Top 3 with Multiple Query Transformation and RRF**

Above I compared two different methods of RAG. The first (baseline) method was a dense vector search that returns the 3 most similar chunks compared to the vectorized query. The second method added a query transformation, which took the original query and called an LLM to produce 3 additional diverse queries similar to the initial one, for 4 total queries. Each of those 4 queries was vectorized and returned a list of the most similar chunks (determined by cosine similarity). Because there are 4 lists of chunk-to-query similarities, RRF is used to rerank the chunks, and from there we derive the top 3.

**Aggregates:**
Multi-query + RRF improved mean per-query recall from 0.757 to 0.907, mean precision from 0.400 to 0.467 (+0.067), and primary-answer hit rate from 90% to 100%.

**Per-query deltas:**
Two queries improved. The Agent Hooks interceptor-timeout query: hits 0 → 1, precision 0 → 0.333, recall 0 → 1.000, primary hit no → yes. The Cosmos DB provider query also gained a hit (precision +0.333, recall +0.5), but its primary chunk was already in the top 3 at baseline, so primary hit stayed yes → yes. The other eight queries were unchanged in this sample.

**Which query types benefited:**
Two patterns showed up. The clearest is a question asking for a specific fact buried in a sentence rather than a named term ("How long does an Agent Hooks interceptor wait before it times out?" → "five seconds"): the baseline's single query didn't phrase things closely enough to the "five seconds" sentence to pull it into the top 3, but one of the three LLM-generated paraphrases must have matched the surrounding wording better, letting RRF surface it. The second is the Cosmos DB query ("Which Cosmos DB provider contains memories of facts or summaries between sessions?"), whose primary chunk was already ranked correctly, but multi-query + RRF pulled in one additional relevant chunk and lifted recall without changing the primary hit.

The other eight queries, which mostly ask for a specific package name, function name, class name, or named provider (e.g., "agent-hooks-sdk", "create_vector_search_tool", "Harness Agent"), were already retrieved correctly by the single-query baseline and saw no change.

**Latency:**
Because there's an LLM call that produces 3 additional queries from 1 LLM output, and then regex parses that output into the 3 separate queries, each of those 4 queries (original + 3 LLM-generated) has to be vectorized and compared against the vectorized chunks. This adds latency, primarily from calling an LLM hosted in Microsoft Foundry. In practice, the multi-query + RRF path averaged about 3.8 seconds per query (median 3.5s), compared to about 0.44 seconds for single-query (median 0.16s) — roughly a 21x slowdown in median latency. Nearly all of that added time comes from the LLM call generating the extra queries.

Since `generate_multiple_queries` calls an LLM, the exact paraphrases — and therefore these exact numbers — can shift somewhat on a rerun; the qualitative pattern (small recall gain, large latency cost) has held across runs.

## Part 3: Find one failure and explain it

### Analysis: No Failure Case

No query regressed in this sample — every query's recall, precision, and hit_primary either improved or stayed exactly the same between single-query and multi-query + RRF. This likely held because with RRF  the original query's ranking is still one of the four lists being fused, so it never gets dropped. With multiple diverse queries I was worried there could be added noise and potentially boost chunks that arent that similar to the original query, but this did not cause any problems in my sample.  A chunk the baseline vector similarity method already ranked in the top 3 keeps enough RRF score from that original ranking alone to stay competitive, so a weak paraphrase can add candidates but can't easily push the correct one out. Since this is pure dense vector search (no BM25/hybrid in this comparison), I'd also note most of my queries target named terms (e.g., "agent-hooks-sdk", "CosmosMemoryContextProvider") which may survive paraphrasing and keep the embedding anchored to the same chunk.


## Part 4: Jev System 1 Model as a Reranker Exploration

This section was inspired by the GitHub repository for RAG Fusion (https://github.com/Raudaschl/rag-fusion/tree/master). That repo includes a Jev reranker experiment (https://github.com/Raudaschl/rag-fusion/tree/master/experiments/jev-in-the-pipeline), MIT licensed, which in turn took inspiration from the Hev reranker repository (https://github.com/hev/reranker/tree/main), Apache 2.0 licensed. I set up a Jev reranker that plugs into my `retrieve()` function.

Jev takes a query and a candidate chunk and returns a single `noul` value — the probability that the answer to a yes/no question is "yes" (0 = no, 1 = yes). Here the question is whether the chunk is relevant to the query, so a `noul` close to 1 means Jev is confident the chunk answers the query, and a value close to 0 means it's confident it doesn't. Given a pool of candidates (either the top-20 from similarity search alone, or the top-20 after RRF fusion), Jev scores every chunk in the pool against the query this way, and the pool is reranked by that probability instead of by cosine similarity or RRF score.

### Jev Experiment 1:
Compare the single return of a vector search that outputs the single most similar chunk and vector search that creates the top pool and have Jev rerank. This is only a dense vector search still. 

In [22]:
# === All outcomes come from text-embedding-3-small embedder ===
embedder = "text-embedding-3-small"

# === Vector Similarity without JEV Reranking for Final K=1 ===
vector_similarity_k1_without_jev = pd.DataFrame(
    score_all_queries_primary_chunk(
        conn,
        retrieval_k=10,
        chunker="md-header",
        final_k=1,
        embedders=embedder,
        hybrid_search=False,
        multiple_query=False,
        jev_rerank=False
    )
)
# === Vector Similarity with JEV Reranking for Final K=1 ===
vector_similarity_k1_with_jev = pd.DataFrame(
    score_all_queries_primary_chunk(
        conn,
        retrieval_k=10,
        pool_k=10,
        final_k=1,
        chunker="md-header",
        embedders=embedder,
        hybrid_search=False,
        multiple_query=False,
        jev_rerank=True,
    )
)

# === Comparison of All Methods for Final K=1 ===
comparison_scores_k1 = pd.concat(
    [vector_similarity_k1_without_jev, vector_similarity_k1_with_jev],
    ignore_index=True,
)


#### Jev Experiment 1: Per Query Deltas

In [23]:
jev_per_query_deltas = compare_retrieval_scores(
    comparison_scores_k1,
    pivot_column="ranking_mode",
    before_value="cosine similarity",
    after_value="Jev",
)
display(
    jev_per_query_deltas[
        [
            "query", "n_hits_delta",
            "precision_delta",
            "recall_delta",
            "hit_primary_before", "hit_primary_after",
        ]
    ].round(3)
)

,query,n_hits_delta,precision_delta,recall_delta,hit_primary_before,hit_primary_after
0,How do I turn on hybrid keyword search for a s...,0,0.0,0.0,True,True
1,How long does an Agent Hooks interceptor wait ...,1,1.0,1.0,False,True
2,What function lets an agent search a vector st...,0,0.0,0.0,False,False
3,What package do I install for the Neo4j GraphR...,0,0.0,0.0,False,True
4,What package do I install to use Agent Hooks?,0,0.0,0.0,True,True
5,What should I pin so evaluation results stay c...,0,0.0,0.0,True,True
6,Which Cosmos DB provider contains memories of ...,0,0.0,0.0,True,True
7,Which agent comes with planning and todo track...,1,1.0,1.0,False,True
8,Which evaluator runs checks locally without ca...,0,0.0,0.0,True,True
9,Which modes of retrieval are supported by Azur...,0,0.0,0.0,True,True


#### Jev Experiment 1: Aggregates

In [24]:
experiment_1_quality_summary_k1 = pd.DataFrame([
    {
        "metric": "Mean precision",
        "before": jev_per_query_deltas["precision_before"].mean(),
        "after": jev_per_query_deltas["precision_after"].mean(),
        "delta": jev_per_query_deltas["precision_delta"].mean(),
    },
    {
        "metric": "Mean recall",
        "before": jev_per_query_deltas["recall_before"].mean(),
        "after": jev_per_query_deltas["recall_after"].mean(),
        "delta": jev_per_query_deltas["recall_delta"].mean(),
    },
    {
        "metric": "Primary-answer hit rate (%, delta in pp)",
        "before": jev_per_query_deltas["hit_primary_before"].mean() * 100,
        "after": jev_per_query_deltas["hit_primary_after"].mean() * 100,
        "delta": jev_per_query_deltas["hit_primary_delta"].mean() * 100,
    },
])

display(experiment_1_quality_summary_k1.round(3))

,metric,before,after,delta
0,Mean precision,0.800,1.000,0.2
1,Mean recall,0.553,0.753,0.2
2,"Primary-answer hit rate (%, delta in pp)",60.000,90.000,30.0


#### Jev Experiment 1: Latency Comparison

In [25]:
latency_columns = [
    "query",
    "retrieval_mode",
    "latency_ms",
]

latency_summary_k1 = (
    comparison_scores_k1
    .groupby(["query_mode", "ranking_mode"], as_index=False)
    .agg(
        mean_ms=("latency_ms", "mean"),
        median_ms=("latency_ms", "median"),
        min_ms=("latency_ms", "min"),
        max_ms=("latency_ms", "max"),
        p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
        p50_ms=("latency_ms", lambda values: values.quantile(0.50)),
        p25_ms=("latency_ms", lambda values: values.quantile(0.25)),
    )
)

display(latency_summary_k1.round(2))

,query_mode,ranking_mode,mean_ms,median_ms,min_ms,max_ms,p95_ms,p50_ms,p25_ms
0,single-query,Jev,364.91,329.66,264.08,606.88,542.46,329.66,310.51
1,single-query,cosine similarity,206.87,192.83,132.92,310.24,309.12,192.83,138.73


#### Analysis: Jev Experiment 1

Baseline (cosine similarity, top 1) vs. the same pool reranked by Jev's P(relevant), top 1.

**Aggregates:**

 Jev reranking improved mean per-query recall from 0.553 to 0.753 (+0.2), mean precision from 0.800 to 1.000 (+0.2), and primary-answer hit rate from 60% to 90%.

**Per-query deltas:** 

Three queries improved — the Agent Hooks interceptor-timeout question, the Neo4j package question (primary hit only, no hit/precision/recall change), and the planning/todo agent question. The other seven were unchanged; none regressed.

**Latency:**

Jev reranking added roughly 70% to median latency, from 192.8ms to 329.7ms (about 1.7x), from the added System One call per query.

### Jev Experiment 2:
Compare single query vector search with Jev Reranker against a Hybrid Search (BM25 + Vector Similarity) with RRF and Jev Reranker

In [26]:
# === All outcomes come from text-embedding-3-small embedder ===
embedder = "text-embedding-3-small"


# === Hybrid Search with JEV Reranking for Final K=1 ===
hybrid_search_multiquery_rrf_with_jev_k1 = pd.DataFrame(
    score_all_queries_primary_chunk(
        conn,
        retrieval_k=10,
        pool_k=20,
        final_k=1,
        chunker="md-header",
        embedders=embedder,
        hybrid_search=True,
        multiple_query=True,
        additional_query_count=3,
        jev_rerank=True,
    )
)

# === Experiment 2 Comparison for Final K=1 ===
experiment_2_comparison_scores_k1 = pd.concat(
    [vector_similarity_k1_with_jev, hybrid_search_multiquery_rrf_with_jev_k1],
    ignore_index=True,
)



#### Jev Experiment 2: Per Query Deltas

In [27]:
jev_2_per_query_deltas = compare_retrieval_scores(
    experiment_2_comparison_scores_k1,
    pivot_column="query_mode",
    before_value="single-query",
    after_value="multi-query",
)
display(
    jev_2_per_query_deltas[
        [
            "query", "n_hits_delta",
            "precision_delta",
            "recall_delta",
            "hit_primary_before", "hit_primary_after",
        ]
    ].round(3)
)

,query,n_hits_delta,precision_delta,recall_delta,hit_primary_before,hit_primary_after
0,How do I turn on hybrid keyword search for a s...,0,0.0,0.0,True,True
1,How long does an Agent Hooks interceptor wait ...,0,0.0,0.0,True,True
2,What function lets an agent search a vector st...,0,0.0,0.0,False,True
3,What package do I install for the Neo4j GraphR...,0,0.0,0.0,True,True
4,What package do I install to use Agent Hooks?,0,0.0,0.0,True,True
5,What should I pin so evaluation results stay c...,0,0.0,0.0,True,True
6,Which Cosmos DB provider contains memories of ...,0,0.0,0.0,True,True
7,Which agent comes with planning and todo track...,0,0.0,0.0,True,True
8,Which evaluator runs checks locally without ca...,0,0.0,0.0,True,True
9,Which modes of retrieval are supported by Azur...,0,0.0,0.0,True,True


#### Jev Experiment 2: Aggregates

In [28]:
experiment_2_quality_summary_k1 = pd.DataFrame([
    {
        "metric": "Mean precision",
        "before": jev_2_per_query_deltas["precision_before"].mean(),
        "after": jev_2_per_query_deltas["precision_after"].mean(),
        "delta": jev_2_per_query_deltas["precision_delta"].mean(),
    },
    {
        "metric": "Mean recall",
        "before": jev_2_per_query_deltas["recall_before"].mean(),
        "after": jev_2_per_query_deltas["recall_after"].mean(),
        "delta": jev_2_per_query_deltas["recall_delta"].mean(),
    },
    {
        "metric": "Primary-answer hit rate (%, delta in pp)",
        "before": jev_2_per_query_deltas["hit_primary_before"].mean() * 100,
        "after": jev_2_per_query_deltas["hit_primary_after"].mean() * 100,
        "delta": jev_2_per_query_deltas["hit_primary_delta"].mean() * 100,
    },
])

display(experiment_2_quality_summary_k1.round(3))

,metric,before,after,delta
0,Mean precision,1.000,1.000,0.0
1,Mean recall,0.753,0.753,0.0
2,"Primary-answer hit rate (%, delta in pp)",90.000,100.000,10.0


#### Experiment 2: Latency Comparison

In [29]:
latency_columns = [
    "query",
    "retrieval_mode",
    "latency_ms",
]

latency_summary_experiment_2 = (
    experiment_2_comparison_scores_k1
    .groupby(["query_mode", "ranking_mode"], as_index=False)
    .agg(
        mean_ms=("latency_ms", "mean"),
        median_ms=("latency_ms", "median"),
        min_ms=("latency_ms", "min"),
        max_ms=("latency_ms", "max"),
        p95_ms=("latency_ms", lambda values: values.quantile(0.95)),
        p50_ms=("latency_ms", lambda values: values.quantile(0.50)),
        p25_ms=("latency_ms", lambda values: values.quantile(0.25)),
    )
)

display(latency_summary_experiment_2.round(2))

,query_mode,ranking_mode,mean_ms,median_ms,min_ms,max_ms,p95_ms,p50_ms,p25_ms
0,multi-query,RRF -> Jev,5554.78,3321.22,2988.17,24169.19,15375.09,3321.22,3201.47
1,single-query,Jev,364.91,329.66,264.08,606.88,542.46,329.66,310.51


#### Analysis: Jev Experiment 2

Single-query vector search + Jev reranking (top 1) vs. hybrid search (BM25 + vector, multi-query, RRF) + Jev reranking (top 1).

**Aggregates:** 

Adding hybrid search and multi-query on top of Jev left mean precision (1.000) and mean recall (0.753) unchanged, but lifted primary-answer hit rate from 90% to 100% (+10pp).

**Per-query deltas:** 

Only the vector-search-tool question changed — its primary hit went from no to yes, with no change to hits, precision, or recall. The other nine queries were unchanged; none regressed.

**Latency:**

The full hybrid + multi-query + Jev pipeline is far slower — median latency rose from 329.7ms to 3321.2ms (about 10.1x), mostly from the added LLM call for query generation plus the extra BM25 and RRF work layered on top of Jev's own call. This path's tail is also notably worse: the max observed latency was ~24.2s and the p95 was ~15.4s, compared to a max of ~0.6s for the single-query + Jev baseline — one query generation or Jev call occasionally stalls, so this pipeline's worst-case latency is far more variable than its median suggests.

In [30]:
conn.close()